## Bronze_To_Silver
### Create By - ABC
### Created Date - 20260519
### Description - contain silvers table information

In [0]:
%run /Workspace/Repos/Project_1705/Azure_Retail_Project/Project_Files/Functions/ACCESS_ADLS_GEN2_USING_SERVICE_PRINCIPALE 


In [0]:
%run /Workspace/Repos/Project_1705/Azure_Retail_Project/Project_Files/Functions/Silver_Layer_Functions

In [0]:
from datetime import datetime
currentdate = datetime.now().strftime("%Y%m%d")
print(currentdate)

In [0]:
dbutils.widgets.text("read_container_name","bronze","Read Container")
read_container_name = dbutils.widgets.get("read_container_name")

In [0]:
dbutils.widgets.text("write_container_name","silver","Write Container")
write_container_name = dbutils.widgets.get("write_container_name")

In [0]:
dbutils.widgets.dropdown("table_name","customers",["customers","orders","order_items","payments","reviews","products","sellers","geolocations","category_translations"],"Table Name")
table_name = dbutils.widgets.get("table_name")

In [0]:
read_base_path = f"abfss://{read_container_name}@retailstorage1881.dfs.core.windows.net"

### Reading data from bronze layer

In [0]:
df1 = spark.read.format('delta').load(f"{read_base_path}/{table_name}/{currentdate}/")
df1.display()

### Rename columns (snake_case)

In [0]:
df1 = snake_case(df1)
df1.display()

### Trim spaces from string columns

In [0]:
df1 = trim_col_values(df1)

In [0]:
from pyspark.sql.functions import to_timestamp

if table_name == "customers":

    # Remove spaces between zip codes
    df1 = remove_spaces(df1, "customer_zip_code_prefix")
  
    # Handle null values
    df1 = df1.na.drop(subset=['customer_id'])
    
    # Remove duplicate records
    df1 = df1.dropDuplicates(["customer_id"])

    # Lowercase customer_city
    df1 = lower_case(df1, "customer_city")

    # Capitalize customer_state
    df1 = initcap_column(df1, "customer_state")

    # Cast data types correctly
    column_dtype_map = {
    "customer_id" : "string",
    "customer_unique_id" : "string",
    "customer_zip_code_prefix" : "int",
    "customer_city" : "string",
    "customer_state" : "string",
    "ingest_ts" : "timestamp"
    }
    df1 = cast_dtype(df1, column_dtype_map)

elif table_name == "orders":  
    # Handle null values
    df1 = df1.na.drop(subset=['order_id'])
    
    # Remove duplicate records
    df1 = df1.dropDuplicates(['order_id'])

    # Cast data types correctly
    column_dtype_map = {
    "order_id" : "string",
    "customer_id" : "string",
    "order_status" : "string",
    "order_purchase_timestamp" : "timestamp",
    "order_approved_at" : "timestamp",
    "order_delivered_carrier_date" : "timestamp",
    "order_delivered_customer_date" : "timestamp",
    "order_estimated_delivery_date" : "timestamp",
    "ingest_ts" : "timestamp"
    }
    df1 = cast_dtype(df1, column_dtype_map)

elif table_name == "order_items": 
    # Remove $ symbol
    df1 = remove_dollar_symbol(df1, "price")
    df1 = remove_dollar_symbol(df1, "freight_value")

    # Cast data types correctly
    column_dtype_map = {
    "order_id" : "string",
    "order_item_id" : "int",
    "product_id" : "string",
    "seller_id" : "string",
    "shipping_limit_date" : "string",  
    "price" : "double",
    "freight_value" : "double",
    "ingest_ts" : "timestamp"
    }
    df1 = cast_dtype(df1, column_dtype_map)
    df1 = df1.withColumn(
    "shipping_limit_date",
    to_timestamp("shipping_limit_date", "dd-MM-yyyy HH:mm"))

    
elif table_name == "payments": 
    # Remove special characters
    df1 = remove_special_chars(df1, "payment_type")

    # Cast data types correctly
    column_dtype_map =  {
    "order_id" : "string",
    "payment_sequential" : "int",
    "payment_type" : "string",
    "payment_installments" : "int",
    "payment_value": "string",
    "ingest_ts" : "timestamp"
    }
    df1 = cast_dtype(df1, column_dtype_map)

    # Remove null and special characters
    df1= remove_null_and_special_chars(df1, "payment_value")

elif table_name == "reviews": 
    # Handle null values
    df1 = df1.na.drop(subset=['review_id'])

    #Remove duplicate records
    df1 = df1.dropDuplicates(["review_id"])

    # Cast data types correctly
    column_dtype_map = {
    "review_id" : "string",
    "order_id" : "string",
    "review_score" : "int",
    "review_comment_title" : "string",
    "review_comment_message": "string",
    "review_creation_date": "timestamp",
    "review_answer_timestamp": "timestamp", 
    "ingest_ts" : "timestamp"
    }
    df1 = cast_dtype(df1, column_dtype_map)

elif table_name == "products":
    # Handle null values
    df1 = df1.na.drop(subset=['prodid'])

    # Remove duplicate records
    df1 = df1.dropDuplicates(["prodid"])

    # Cast data types correctly
    column_dtype_map = {
    "prodid" : "string",
    "productcattegory_name" : "string",
    "productnamlenght" : "int",
    "product_description_lenght" : "int",
    "productphotosqty" : "int",
    "productweight_g" : "int",
    "prodlenghth" : "int",
    "product_height_cm" : "int",
    "product_width_cm" : "int",
    "ingest_ts" : "timestamp"
    }
    df1 = cast_dtype(df1, column_dtype_map)

elif table_name == "sellers":
    # Handle null values
    df1 = df1.na.drop(subset=['seller_id'])

    # Remove duplicate records
    df1 = df1.dropDuplicates(["seller_id"])

    # Cast data types correctly
    column_dtype_map = {
    "seller_id" : "string",
    "sellerzip_code_prefix" : "int",
    "seller_city" : "string",
    "seller_state" : "string",
    "ingest_ts" : "timestamp"
    }
    df1 = cast_dtype(df1, column_dtype_map)

elif table_name == "geolocations":
    # Cast data types correctly
    column_dtype_map = {
    "geolocation_zip_code_prefix" : "int",
    "geolocation_lat" : "double",
    "geolocation_lng" : "double",
    "geolocation_city" : "string",
    "geolocation_state" : "string",
    "ingest_ts" : "timestamp"
    }
    df1 = cast_dtype(df1, column_dtype_map)

elif table_name == "category_translations":
    # Cast data types correctly
    column_dtype_map = {
    "product_category_name" : "string",
    "product_category_name_english" : "string",
    "ingest_ts" : "timestamp"
    }
    df1 = cast_dtype(df1, column_dtype_map)

df1.printSchema()
df1.display()


      

### Writing data in silver layer

In [0]:
write_base_path = f"abfss://{write_container_name}@retailstorage1881.dfs.core.windows.net"

In [0]:
df1.write.format("delta").mode("overwrite").option("header","true").save(f"{write_base_path}/{table_name}/{currentdate}/")